# Carbon fate

In [34]:
import sys
sys.path.append('/projects/code/icesat2_boreal/lib')
import os
import geopandas as gpd
import pandas as pd
import mosaiclib

In [2]:
import carbon_fate

In [54]:
hybas_fn = '/projects/shared-buckets/montesano/databank/hydrobasins/hybas_lake_ar_lev08_v1c.shp'

In [5]:
FOCAL_TILE = 3457

In [35]:
tindex_cacc = pd.read_csv(mosaiclib.MISC_TINDEX_FN_DICT['CACC_2020_v3.1_multiyr_nsims050'])
cacc_fn = tindex_cacc[tindex_cacc.tile_num.isin([FOCAL_TILE])].s3_path.to_list()[0]
cacc_fn

's3://maap-ops-workspace/montesano/dps_output/run_carbon_accumulation/carbon_accumulation_v3/CACC_2020_v3.1_multiyr_nsims050/2025/10/03/11/55/01/720342/boreal_cacc_2020_0003457.tif'

In [36]:
#### Eventually, use the 5-yr trends from 2021-2025

In [37]:
biomass_trends_classes_fn = '/projects/my-public-bucket/local_output/compute_trends/full_run_niter30_multiyear_atl08v6/boreal_agb_trends_0003457_10yrs_ols_kendallclasses.tif'

In [38]:
outdir = '/projects/my-public-bucket/local_output/carbon_fate'
!mkdir -p $outdir

In [10]:
hybas = gpd.read_file('/projects/shared-buckets/montesano/databank/hydrobasins/hybas_lake_ar_lev08_v1c.shp')

In [11]:
detailed_df = pd.read_csv('/projects/my-public-bucket/local_output/carbon_fate/detailed_results.csv')
basin_ids_for_tile = detailed_df.basin_id.to_list()
len(basin_ids_for_tile)

25

In [12]:
boreal_tiles = gpd.read_file(mosaiclib.boreal_tile_index_path)

In [19]:
hybas_filt = carbon_fate.filter_intersecting_hydrobasins_spatial_index(hybas, biomass_trends_classes_fn, buffer_degrees=0.0001, verbose=False)
hybas_filt.shape

(36, 16)

In [18]:
hybas_filt.explore(m=boreal_tiles[boreal_tiles.tile_num.isin([FOCAL_TILE])].explore(color='red'))

In [20]:
hybas_filt[hybas_filt.HYBAS_ID.isin(basin_ids_for_tile)].explore(m=boreal_tiles[boreal_tiles.tile_num.isin([FOCAL_TILE])].explore(color='red'))

In [50]:
args = f'--carbon-raster {cacc_fn} '
args += f'--kendall-classes {biomass_trends_classes_fn} '
args += f'--hydrobasins {hybas_fn} '
args += f'--age-band 9 '
args += f'--class-band 1 '
args += f'--output {outdir} '
args += '--verbose' # --use-vrt'
args

'--carbon-raster s3://maap-ops-workspace/montesano/dps_output/run_carbon_accumulation/carbon_accumulation_v3/CACC_2020_v3.1_multiyr_nsims050/2025/10/03/11/55/01/720342/boreal_cacc_2020_0003457.tif --kendall-classes /projects/my-public-bucket/local_output/compute_trends/full_run_niter30_multiyear_atl08v6/boreal_agb_trends_0003457_10yrs_ols_kendallclasses.tif --hydrobasins /projects/shared-buckets/montesano/databank/hydrobasins/hybas_lake_ar_lev08_v1c.shp --age-band 9 --class-band 1 --output /projects/my-public-bucket/local_output/carbon_fate --verbose'

In [51]:
cmd = f'/opt/conda/envs/pangeo/bin/python /projects/code/icesat2_boreal/lib/carbon_fate.py {args}'

In [52]:
%%time
!eval $cmd

Starting cohort-trend fate analysis using Kendall Tau classes...
Carbon raster: s3://maap-ops-workspace/montesano/dps_output/run_carbon_accumulation/carbon_accumulation_v3/CACC_2020_v3.1_multiyr_nsims050/2025/10/03/11/55/01/720342/boreal_cacc_2020_0003457.tif
Kendall classes raster: /projects/my-public-bucket/local_output/compute_trends/full_run_niter30_multiyear_atl08v6/boreal_agb_trends_0003457_10yrs_ols_kendallclasses.tif
Hydrobasins: /projects/shared-buckets/montesano/databank/hydrobasins/hybas_lake_ar_lev08_v1c.shp
Output directory: /projects/my-public-bucket/local_output/carbon_fate

Kendall Tau class mapping:
  0: No data -> Ignored
  1: Strong sig. positive -> Strong increase
  2: Moderate sig. positive -> Strong increase
  3: Weak sig. positive -> Moderate increase
  4: Very weak sig. positive -> Moderate increase
  5: Non-sig. positive -> Stable
  6: Non-sig. negative -> Stable
  7: Very weak sig. negative -> Moderate decline
  8: Weak sig. negative -> Moderate decline
  9: M

In [53]:
outdir = '/projects/my-public-bucket/local_output/carbon_fate_landscapes'
!mkdir -p $outdir

In [55]:
cmd = f'/opt/conda/envs/pangeo/bin/python /projects/code/icesat2_boreal/lib/carbon_fate_landscapes.py -h'
!eval $cmd

usage: carbon_fate_landscapes.py [-h] --carbon-raster CARBON_RASTER
                                 --kendall-raster KENDALL_RASTER --polygons
                                 POLYGONS [--age-band AGE_BAND]
                                 [--kendall-band KENDALL_BAND]
                                 [--batch-size BATCH_SIZE]
                                 [--n-processes N_PROCESSES]
                                 [--polygon-start POLYGON_START]
                                 [--polygon-count POLYGON_COUNT] --output
                                 OUTPUT [--output-prefix OUTPUT_PREFIX]
                                 [--age-config AGE_CONFIG]
                                 [--polygon-id-col POLYGON_ID_COL]
                                 [--polygon-area-col POLYGON_AREA_COL]
                                 [--use-vrt] [--verbose] [--debug]

Analyze forest age cohort fate based on biomass trends with batch processing

options:
  -h, --help            show this help message

In [58]:
mosaiclib.LC_MOSAIC_JSON_FN_DICT['c2020updated']

's3://maap-ops-workspace/shared/montesano/DPS_tile_lists/run_build_stack/build_stack_v2023/LC_ESA_WC_2020/LC_tindex_master_mosaic.json'

In [79]:
# Debug mode with single process
cmd = f'/opt/conda/envs/pangeo/bin/python /projects/code/icesat2_boreal/lib/carbon_fate_landscapes.py \
    --carbon-raster {mosaiclib.MISC_MOSAIC_JSON_FN_DICT['CACC_2020_v3.1_multiyr_nsims050']} \
    --kendall-raster {mosaiclib.LC_MOSAIC_JSON_FN_DICT['c2020updated']} \
    --age-band 9 \
    --kendall-band 1 \
    --polygons {hybas_fn} \
    --polygon-count 10 \
    --n-processes 1 \
    --batch-size 5 \
    --debug \
    --verbose \
    --output {outdir}'

In [80]:
!eval $cmd

Starting cohort fate analysis with batch processing...
Carbon raster: s3://maap-ops-workspace/shared/montesano/DPS_tile_lists/CACC/carbon_accumulation_v3/CACC_2020_v3.1_multiyr_nsims050/CACC_tindex_master_mosaic.json
Kendall raster: s3://maap-ops-workspace/shared/montesano/DPS_tile_lists/run_build_stack/build_stack_v2023/LC_ESA_WC_2020/LC_tindex_master_mosaic.json
Polygons: /projects/shared-buckets/montesano/databank/hydrobasins/hybas_lake_ar_lev08_v1c.shp
Batch size: 5
Processes: 1
Loading polygons from: /projects/shared-buckets/montesano/databank/hydrobasins/hybas_lake_ar_lev08_v1c.shp
Total polygons in file: 15551
Selected polygons 0 to 9 (10 total)
Processing 10 polygons with 1 processes
Batch size: 5
Created 2 batches for processing
Mosaic inputs detected - Carbon: True, Kendall: True
🔧 Worker batch_1: Processing 5 polygons...
🔧 Worker batch_1: Using mosaic mode with imported MosaicHandler
🔧 Worker batch_1: Initialized S3 manager
🔧 Loading mosaic JSON from S3: s3://maap-ops-worksp